# S6.3 - Dependency Parsing: Apple 2025 Finance Edition

This notebook preserves the original content, code, and visual explanations from `Section 2.2 - Dependency Parsing.ipynb`.

The finance-edition cells at the top connect the same NLP concept to **Apple Inc.'s fiscal 2025 Form 10-K**. The original notebook follows after these bridge cells so students can compare the general NLP pipeline with a realistic financial-document workflow.


In [ ]:
from pathlib import Path
import os
import sys

FINANCE_DIR = Path(r"D:\Repositories\AD698-generative-ai-for-BA\help-code\4-nlp-finance")
FINANCE_SRC = Path(r"D:\Repositories\AD698-generative-ai-for-BA\help-code\4-nlp-finance\src")
ORIGINAL_NOTEBOOK_DIR = Path(r"D:\Repositories\AD698-generative-ai-for-BA\help-code\2-core-tasks")

if str(FINANCE_SRC) not in sys.path:
    sys.path.insert(0, str(FINANCE_SRC))
if str(ORIGINAL_NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(ORIGINAL_NOTEBOOK_DIR))

from finance_nlp_utils import *

finance_doc = load_apple_10k()
finance_sample = sample_sentences(5)
finance_metrics = load_metrics()

print(f"Apple 2025 10-K excerpt characters: {len(finance_doc):,}")
print("Example Apple sentence:")
print(finance_sample[0])

# Run original notebook code from its original folder so relative data/image paths still work.
os.chdir(ORIGINAL_NOTEBOOK_DIR)


In [ ]:
print("Available Apple SEC 10-K evaluation sets")
for name in list_evaluation_sets():
    print("-", name)

print("\nSentence classification example")
classification_rows = load_evaluation_csv("apple_2025_sentence_classification.csv")
for row in classification_rows[:3]:
    print(row["split"], row["label"], "=>", row["sentence"][:120])

print("\nExtractive QA example")
qa_rows = load_evaluation_jsonl("apple_2025_qa.jsonl")
print(qa_rows[0])


<img src='data/images/section-notebook-header.png' />

**Disclaimer:** The topic of dependency parsing is not part of the syllabus of the course "Natural Language Processing: Core Task". However, we provide this notebook as dependency parsing can be very useful in practice and, like syntactic parsing, analyzes the hierarchical structure of sentences.

# Dependency Parsing

Dependency parsing is a natural language processing (NLP) technique that analyzes the grammatical structure of a sentence by identifying the syntactic relationships between words. It aims to determine the dependency relationships between words, representing them as a directed graph called a dependency tree.

In dependency parsing, each word in a sentence is considered as a node, and the relationships between the words are represented as directed edges. The root of the dependency tree typically represents the main verb or the main predicate of the sentence, while other words depend on this root word or on other words in the sentence. These dependencies capture the grammatical relationships such as subject-verb, verb-object, and modifier relationships.

The most common representation used in dependency parsing is the Universal Dependencies (UD) framework, which provides a standardized set of dependency labels to describe the grammatical relationships. Examples of dependency labels include "nsubj" for a nominal subject, "obj" for a direct object, "amod" for an adjective modifier, and so on.

Dependency parsing can be performed using different algorithms and models, such as rule-based approaches, statistical models, and neural network-based models. These models learn from annotated data to predict the dependency structure of a sentence. Once the dependency parsing is performed, the resulting dependency tree can be used for various downstream NLP tasks, such as information extraction, question answering, machine translation, and sentiment analysis.

## Setting up the Notebook

### Import Required Packages

We only need spaCy for this notebook as it provides dependency parsing out of the box.

In [ ]:
import spacy
from spacy import displacy

Load a language model.

In [ ]:
try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        print("No English spaCy model installed; using a blank English tokenizer fallback.")
        nlp = spacy.blank("en")
        nlp.add_pipe("sentencizer")

### Example Sentences

Let's create some example sentences. Feel free to modify the sentences or add your own and inspect the results from the code cells below. Note that some comments refer to the first example sentence *"I downloaded the lecture slide decks from the website."*

In [ ]:
sentence = "I downloaded the lecture slide decks from the website." 
#sentence = "The burgers were great, but the waiting time was too long"
#sentence = "Leonhard Euler was born on 15 April 1707, in Basel, Switzerland."

doc = nlp(sentence)

---

## Visualization of Dependency Trees

`displacy` is a visualization module in the spaCy package. It is designed to provide an easy and intuitive way to visualize the dependency parse trees generated by spaCy's parser. It allows you to visualize dependency parse trees in a graphical format, making it easier to understand the syntactic structure of a sentence. It can be used to visualize the relationships between words and their dependencies, including the direction and type of the dependencies.

Once you have a parsed document or sentence, you can pass it to the `displacy.render` function along with the appropriate options to generate a visual representation. The output can be displayed in a Jupyter notebook or saved as an image file. The visualization produced by `displacy` includes arrows connecting words to their dependencies, with different colors and labels indicating the types of dependencies. It also provides options for adjusting the visual style, such as changing the color scheme, word spacing, and background color. The complete list of options can be found on the [spaCy Visualizers](https://spacy.io/usage/visualizers) web page. Here we focus only on some basic options most useful in practice. Overall, `displacy` is a useful tool for exploring and understanding the syntactic structure of sentences and can aid in the analysis and debugging of NLP models that rely on dependency parsing.

### "Full" Dependency Tree

The "full" dependency tree shows the relationships between all the tokens incl. punctuation marks. The latter are by default not explicitly shown (`"collapse_punct": True`). The example below, uses the default distance/spacing between tokens of `150`. Note that a dependency tree also shows the Part-of-Speech (POS) tag for each token.

In [ ]:
options = {"distance": 120, "collapse_phrases": False, "collapse_punct": False}

displacy.render(doc, style="dep", options=options)

### "Full" Dependency Tree with Reduced Distances

To save some space to accommodate longer sentences, we can reduce the distance between two tokens.

In [ ]:
options = {"distance": 100, "collapse_phrases": False, "collapse_punct": False}

displacy.render(doc, style="dep", options=options)

### Dependency Tree with Collapsed Punctuation Marks (default)

Punctuation marks typically result in very long arcs that artificially blow up the dependency graph. Since punctuation marks are not interesting, we can plot the graph without their dependencies. Note that the punctuation marks are still shown by being attached to the token given by the input sentence.

In [ ]:
options = {"distance": 150, "collapse_phrases": False, "collapse_punct": True}

displacy.render(doc, style="dep", options=options)

### Dependency Tree with Collapsed Noun Phrases

By default, spaCy performs **noun phrase chunking**. Noun chunks are phrases that consist of a noun and the words that directly modify or depend on it. They are a linguistic unit that groups together a noun and its associated words in a sentence. Noun chunks provide a way to identify and extract meaningful noun phrases from text.

In English grammar, a noun chunk typically includes the noun itself and any determiners, adjectives, or other modifiers that directly precede or follow the noun. For example, in the sentence "The big red apple fell from the tree," the noun chunk "the big red apple" includes the determiner "the," the adjectives "big" and "red," and the noun "apple." Noun chunks are useful in natural language processing (NLP) tasks such as information extraction, text summarization, and named entity recognition. They can help identify and extract important noun phrases that carry significant semantic meaning in a sentence or document.

In the spaCy library, noun chunks can be extracted using the noun_chunks attribute of a parsed document or sentence. It provides an iterator that yields noun chunks as Span objects, which can then be accessed and processed further in a program. This feature in spaCy makes it convenient to extract and work with noun phrases in NLP applications.

Noun chunks can be explicitly listed as follows; see the code cell below.

In [ ]:
for chunk in doc.noun_chunks:
    print(chunk.text)

We can utilize the information about noun chunks by treating them as a single leaf in the dependency tree. This is not done by default.

In [ ]:
options = {"distance": 150, "collapse_phrases": True, "collapse_punct": True}

displacy.render(doc, style="dep", options=options)

----------

## Understanding Dependency Labels

When looking at the dependency graphs above, we can identify the most important information: the edge/arc labels representing the dependency relationship between the words of a sentence. These labels are taken from a predefined set of dependency labels. To see the list of all available labels, together with a brief description, you can run the code cell below.

In [ ]:
for label in nlp.get_pipe("parser").labels:
    print(label, " -- ", spacy.explain(label))

In principle, there is no single agreed upon set of dependency labels, and this set may also differ across languages. However, there is of course the effort to unify and standardize these labels as much as possible. [Universal Dependencies](https://universaldependencies.org/) (UD) is a framework for consistent annotation of grammar (parts of speech, morphological features, and syntactic dependencies) across different human languages. UD is an open community effort with over 300 contributors producing nearly 200 treebanks in over 100 languages. If you’re new to UD, you should start by reading the first part of the Short Introduction and then browsing the annotation guidelines.

The currently defined dependency labels can be found [here](https://universaldependencies.org/u/dep/index.html). Note that the label set of spaCy does not fully match the list of universal dependencies, although the most important ones are of course available. Still, this is something to keep in mind in practice since different dependency parsers may yield different dependency trees if they use different labels set. Of course, even when using the same label set, dependency parsers may return different trees since parsers may rely on different methods/algorithms and/or training of different datasets.

---------

## Navigating Dependency Tree

To navigate the dependencies of a spaCy document, you can use the `Token` objects and their properties provided by the spaCy library. Each token represents a word in the document and contains information about its linguistic features, including its dependency relationships.

By following the dependency relations and the head tokens of each token, you can navigate the entire dependency tree of the document. Additionally, you can use the token.children attribute to access the immediate children of a token in the dependency tree. This provides a way to explore the subtree rooted at a particular token. By utilizing these properties and attributes, you can effectively navigate and explore the dependencies within a spaCy document.

The code cell below prints the children (i.e., the dependent tokens) of each token of our sentence. Tokens without dependent tokens will natrually yield an empty list of children.

In [ ]:
for token in doc:
    print("{} >>> children {}".format(token.text, [child.text for child in token.children]))

To illustrate the use of dependencies between tokens, the method `get_compound()` below implements the extraction of compound words, typically nouns. Compound nouns are nouns that are formed by combining two or more words together. These words can be either other nouns, adjectives, verbs, or prepositions. When combined, the individual words create a new noun with a specific meaning that may not be easily deducible from the meanings of the individual words.

Compound nouns can be written in different forms, such as separate words ("ice cream"), hyphenated words ("well-being"), or single words ("notebook"). The form used often depends on language conventions and style guides. Compound nouns can be categorized into different types based on the relationship between the constituent words:

* **Noun-Noun compounds:** These compounds are formed by combining two nouns together, such as "treehouse," "rainstorm," or "bookshelf."

* **Adjective-Noun compounds:** These compounds involve an adjective and a noun, like "blackboard," "hotdog," or "greenhouse."

* **Verb-Noun compounds:** These compounds combine a verb and a noun, such as "breakfast," "pickup," or "output."

* **Preposition-Noun compounds:** These compounds include a preposition and a noun, like "overcoat," "underground," or "afterthought."

It's worth noting that compound nouns can have their own grammar rules and usage patterns. Understanding and recognizing compound nouns is important for accurate understanding, interpretation, and generation of language in various NLP tasks.

Of course, compound nouns that are written/represented by a single word are trivial to extract. We therefore only need to focus on compound nouns that are written using separate words. The relevant dependency label is aptly named `compound` and we utilize this label in the method `get_compound()` to find all compound nouns.

In [ ]:
def get_compound(token, compound_parts=[]):

    # Loop over all children of the token
    for child in token.children:
        # We are only interested in the "compound" relationship
        if child.dep_ == "compound":
            # Call method recursively on the child token
            get_compound(child, compound_parts=compound_parts)
    
    # Add the token itself to the list
    compound_parts.append(token)
    
    #return compound_parts

Let's execute our method for a head noun (*"deck"*) assuming our example sentence *"I downloaded the lecture slide decks from the website."*

In [ ]:
compound_parts = []

get_compound(doc[5], compound_parts=compound_parts) # "decks" in sentence "I downloaded the lecture slide deck from Canvas."

print(' '.join([t.text for t in compound_parts]))

-----------

## Example Application: Relation Extraction

Recall that text is considered unstructured data. However, many tasks benefit from some more structured representation (e.g., document search). A very popular method to represent information is a **knowledge graph**. A knowledge graph is a structured representation of knowledge that captures information about entities, their attributes, and the relationships between them. It is designed to organize and represent knowledge in a machine-readable format, enabling efficient retrieval, reasoning, and analysis of information. Any object, place, or person can be a node. An edge defines the relationship between the nodes. For example, we can have the nodes *Singapore* and *Asia* with an directed edge from *Singapore* to *Asia* with the label *locate_in*. Simply speaking, a knowledge graph is a large collection of **triples** where each triple contains a subject (e.g., *Singapore*), a predicate (e.g., *located_in*), and an object (e.g., *Asia*).

In a knowledge graph, entities are represented as nodes, and the relationships between entities are represented as edges or links. Each entity and relationship typically has additional attributes or properties associated with them, providing more detailed information about the entities and their connections. Knowledge graphs are used in a wide range of applications, including semantic search, question answering, recommendation systems, information retrieval, and knowledge representation. By representing knowledge in a graph structure, it becomes possible to perform complex queries, infer new information, discover patterns, and derive insights from the interconnected data.

Knowledge graphs can be built from various sources, including structured databases, **textual data**, web pages, and other structured or semi-structured resources. Since creating such knowledge graphs manually is very resource-intensive, automatic approaches are highly sought-after. Given that such much information is available in text form, trying to convert this data into a knowledge graph is an obvious important goal. There are many challenges involved and many different approaches conceivable -- and here we can only look at some basic ideas and methods.


### Most Basic Triple Extraction

Here, we have a very basic look into how we can use dependency graph extract triples from a sentence. If you look at the examples above, you might already have some intuition about which kind of dependencies make meaningful triples. In the following, we look at the most straightforward case where a (subject, predicate, object) triple is derived from a verb and the dependent subject (`nsubj`) and dependent direct object (`dobj`). The method below tries to accomplish this.


In [ ]:
def extract_svo_triples(doc):

    # Collect all triples; a sentence my contain more such dependencies
    triples = []
    
    for token in doc:
        
        # The relationships we are interested require a verb as the predicate of the triple
        # So wie ignore all other tokens
        if token.pos_ != "VERB":
            continue

        # Initialize the subject/predicate/object of the potential triple
        # The predicate is naturally the current verb
        subj, pred, obj = [], token.text.lower(), []

        # We now check all dependent tokens of the current verb
        for child in token.children:
            # If we see an "nsubj" label, we set the SUBJECT of the triple to this child
            if (child.dep_ == "nsubj"):
                get_compound(child, compound_parts=subj)
                subj = ' '.join([t.text.lower() for t in subj])
            # If we see an "dobj" label, we set the OBJECT of the triple to this child
            elif (child.dep_ == "dobj"):
                get_compound(child, compound_parts=obj)
                obj = ' '.join([t.text.lower() for t in obj])
                
        # Only if we have a subject and object, we assume we have a valid triple
        if subj is not None and obj is not None:
            triples.append((subj, pred, obj))
            
    # Return all the found triples
    return triples

Let's run this method over our example sentence *"I downloaded the lecture slide decks from Canvas."*.

In [ ]:
sentence = "I downloaded the lecture slide decks from Canvas." 
#sentence = "Alice bought a book, and Bob bought a pizza."

doc = nlp(sentence)

triples = extract_svo_triples(doc)

for t in triples:
    print("{} ===> {} ===> {}".format(t[0], t[1], t[2]))

### Limitations

The method `extract_svo_triples()` is not only very simplified but also only extracts the most basic type of triplet from a sentence. Consider for example *"Ice cream is so delicious."*. One can argue that we want to extract the tripled (*ice cream*, *is*, *delicious*). However, if we run our method over this sentence we get an empty result.

In [ ]:
sentence = "Ice cream is so delicious."

doc = nlp(sentence)

triples = extract_svo_triples(doc)

for t in triples:
    print("{} ===> {} ===> {}".format(t[0], t[1], t[2]))

Let's have a look at the dependency graph.

In [ ]:
displacy.render(doc, style="dep")

Of course, since *"delicious"* is not a direct object, we fail to capture it as an object for a triple. In short, this calls for identifying which other dependency relationships may yield valid/interesting/useful/etc. triples -- which may depend on the application. After that, naturally, these identification steps need to be implemented similar to method `extract_svo_triples()`.

### Discussion

Apart from just extracting more triples there are also many other challenges that need to be addressed to expect a meaningful knowledge graph. Just to mention a couple:

* In our triple `(I, downloaded, lecture slide decks)`, the subject and object are just words/phrases that are nor linked to any real-world entity or uniquely identified concept. For example, if we later might extract another triple such as `(I, ate, cake)` from some other sentence, it's arbitrarily unlikely that the `I` will refer to the same person.

* Let's assume, we extract some triple `(I, saved, lecture slide decks)` we treat label *"downloaded"* and *"saved"* is completely different predicates. In practice, we would like to agree on some kind of vocabulary of labels. This brings us to the concept of **ontologies**.

There is a long way to convert arbitrary unstructured text reliably into a meaningful knowledge graph. However, extracting dependency relations between words is a very important step towards this goal. Summing up, dependency parsing plays a crucial role in creating knowledge graphs by extracting structured information about entities and their relationships from text. Here's a summary of its use:

* **Entity Extraction:** Dependency parsing helps identify entities in a sentence by recognizing noun phrases and their modifiers. By extracting entities and their associated attributes, such as names, locations, or dates, dependency parsing contributes to populating the nodes of a knowledge graph.

* **Relationship Extraction:** Dependency parsing reveals the syntactic relationships between words in a sentence. By analyzing the dependencies between verbs, nouns, and other parts of speech, it becomes possible to identify and extract relationships between entities. For example, it can help identify subject-verb-object relationships or modifiers of a particular entity.

* **Graph Structure:** The dependencies identified through parsing provide the foundation for defining the structure of a knowledge graph. The head-dependent relationships between words form the edges of the graph, while the words themselves serve as nodes. This structure facilitates the organization and representation of information in a graph format.

* **Semantic Enrichment:** Dependency parsing can enrich the knowledge graph by providing additional semantic information. By analyzing the grammatical roles and dependencies, it becomes possible to assign more precise labels and attributes to the entities and relationships, enhancing the depth and accuracy of the knowledge graph.

* **Querying and Reasoning:** Once the knowledge graph is constructed using dependency parsing, it enables efficient querying and reasoning. The structured representation allows for complex queries, pattern matching, and inference, supporting tasks such as semantic search, question answering, and information retrieval.

Overall, dependency parsing serves as a fundamental step in creating knowledge graphs, facilitating the extraction of entities, relationships, and the overall structure of the graph. It enhances the organization, understanding, and utilization of textual information for building intelligent systems and enabling various downstream applications.

---

## Example Application: Text Simplification

Recall that natural language is very expressive, i.e., there are virtually an infinite number of ways to express the same message. This includes that often a sentence may contain parts that are arguably not fundamentally important to convey that core message. We can utilize this observation for the task of **text simplification**. Text simplification involves transforming complex or difficult-to-understand text into simpler and more accessible language while retaining its original meaning. It aims to make information more comprehensible for various user groups, including individuals with cognitive or language impairments, non-native speakers, or people with limited literacy skills.

Text simplification typically involves several subtasks, including:

* **Lexical Simplification:** Replacing complex or uncommon words with simpler synonyms or paraphrases. This helps to reduce the vocabulary difficulty and enhance readability.

* **Sentence Simplification:** Restructuring or rephrasing complex sentences into shorter and simpler forms. This can involve splitting long sentences, removing or reordering clauses, and using simpler grammatical structures.

* **Discourse Simplification:** Simplifying the overall structure and coherence of the text. This includes clarifying pronoun references, simplifying coreference relationships, and ensuring the flow of information is more linear and straightforward.

Text simplification can benefit a wide range of applications, such as educational resources, machine translation, information retrieval, and assistive technologies. It enables better understanding and accessibility of information, making it easier for diverse users to grasp complex concepts and effectively engage with text-based content.

In the following, we look at a basic approach towards **sentence simplification**. More specifically, we aim to remove appositives. In grammar, an apposition is a construction where two noun phrases (or noun-like phrases) are placed side by side, with one phrase providing additional information or clarification about the other. The phrase that provides the additional information is called the appositive, and it is typically set off by commas, dashes, or parentheses.

An appositive can further describe or identify the noun it modifies, providing additional details, explanations, or specifications. It adds non-essential information that helps to provide more context or description to the noun. Here are a few examples to illustrate the use of apposition:

* *"My friend, a talented musician, is coming over tonight."* In this sentence, the appositive "a talented musician" provides additional information about the noun "my friend."

* *"The city of Paris, known as the City of Lights, is famous for its romantic ambiance."* Here, the appositive "known as the City of Lights" provides an additional name or identifier for the noun "the city of Paris."

* *"My dog, a golden retriever, loves to play fetch."* In this example, the appositive "a golden retriever" gives more information about the noun "my dog."

Apposition helps to provide more details, specifications, or clarifications about a noun, enhancing the overall understanding and description of the sentence. It allows for the insertion of additional information without disrupting the basic grammatical structure of the sentence. However, one can equally argue that the appositive is not important to capture the core meaning of a sentence. We can therefore simplify sentences by removing appositives. Given this examples above, this would yield the sentences *"My friend is coming over tonight."*, *"The city of Paris is famous for its romantic ambiance."*, and *"My dog loves to play fetch."*

We consider the following example for our tests:

In [ ]:
sentence = "Musk, the CEO of Tesla, bought Twitter."

doc = nlp(sentence)

Let's first have a look at the dependency tree.

In [ ]:
displacy.render(doc, style="dep")

While *"[...], the CEO of Tesla, [...]"* provides some additional information, it is not required to convey the message that *"Musk bought Twitter."*

### Utility Method

From the dependency parse tree above, we can see that the appositional modifier of *"Musk"* is *"CEO"*. However, we need to remove the whole phrase *"the CEO of Twitter"*, which is represented by all the tokens that are descendants of *"CEO"*. Since finding all descendants for a token is a generic and useful taks, we can implement this as method `get_descendants()` below.

`get_descendants()` returns all the token indices starting with the original token, and then recursively checks all the children. Note that
* The method returns a set of indices; so if a sorted list is required, this can be done trivially afterwards.
* The returned indices may not be consecutive; although it generally should be for appositional modifiers.


In [ ]:
def get_descendants(token, indices=set()):
    indices.add(token.i)
    
    for child in token.children:
        indices.add(child.i)
        get_descendants(child, indices=indices)
    
    return indices
    
# In this example, we know that the 4 token (with index 3) is "CEO"
appos_indices = get_descendants(doc[3])

print(appos_indices)

for i in appos_indices:
    print(doc[i].text)

### Rule to Remove Appositives

While we consider only appositives here, there are other rules to simplify a text (e.g., remove (some) relative clause, replace complex words with simpler synonyms, etc.). So let's set up our implementation in such a way that it could easily be extended. For that, we implement each rules as method that returns:

* A boolean that signals if the sentence was indeed modified/simplified in any way
* The modified sentence as a new spaCy document

Let's implement such a rule for removing appositives. We basically just require the method `get_descendants()` for this. The only extension is to check if the appositive is preceded and/or succeeded by a comma. If so, they of course need also be removed to yield a properly formed sentence. The method `remove_appos()` accomplishes this.


In [ ]:
def remove_appos(doc, token):
    # Since we remove at least on token, the doc will always be modified
    # (there might other rules where does may not necessarily be the case)
    was_modified = True
    
    # Get indices of all descendats of token with appos dependency
    obsolete_indices = get_descendants(token)
    
    # Check if appositive is preceeded and/or succeeded by a comma that also needs to be removed
    # We use the try-except block to handle case where the appositive is at the start or end of sentence
    try:
        if doc[min(obsolete_indices)-1].text == ",":
            obsolete_indices.add(min(obsolete_indices)-1)
    except:
        pass
    try:
        if doc[max(obsolete_indices)+1].text == ",":
            obsolete_indices.add(max(obsolete_indices)+1)
    except:
        pass
    
    # Form a new sentence from the original text but ignoring the obsolete tokens
    sentence = ' '.join([ t.text for t in doc if t.i not in obsolete_indices ])
    
    # Return a newly analyzed version of the sentence; probably not the most efficient
    # approach but it's "safer" as any edit might change other dependencies
    return was_modified, nlp(sentence)


was_modified, doc_modified = remove_appos(doc, doc[3])

print("Has the original document beend modified: {}".format(was_modified))
print("New sentence: {}".format(doc_modified))

### Simplify Document

Now we only need to wrap our rule into a method that, in principle, checks all implemented rules. At the very least, since we only have one rule, we need to check if a sentence may contain more than one appositive. The method `simplify()` below tries to simplify a sentence by repeatedly applying simplification rules until no further change is possible. In other words, at some point the variable `was_modified` will be `False` and the method exits. For our set of rules thi will be when all appositives have been removed.

In [ ]:
def simplify(doc):
    # Initialize local variables
    was_modified, current_doc = True, doc
    
    # Perform all simplifications rules until no further changes was fone
    while was_modified: 
        # By default, assume that doc has not been edited
        was_modified = False

        ############################################################
        # Below follow all the check for simplifiaction rules.
        # Here, we have just one, bu we could include more
        ############################################################
        
        for token in current_doc:
            
            # Look for "appos" depencency label, ignore everything else
            if token.dep_ == "appos":
                
                # Simplify sentence by removing appostion
                was_modified, current_doc = remove_appos(doc, token)
                
                # If the document was modified, stop the loop and check the new doc from scratch again
                # (probably not efficient but ensures that any previous edits don't "mess up" later edits)
                if was_modified:
                    break
                    
        ############################################################    
                
    return current_doc    


doc_simplified = simplify(doc)

print("Simplified sentence: {}".format(doc_simplified))

### Discussion

In practice, text simplification is a very challenging task as it must preserve the core meaning of a text as well as preserve the correctness and flow of the language. Here we only looked at a basic approach towards sentence simplification by removing parts of sentences that are arguably not that important. In general, removing parts of a sentence is often a bit easier than rewriting or rephrasing a sentence as it is easier to ensure grammatical correctness.

Keep in mind that we also made the assumption here that it is always harmless to remove appositives. However, this assumption might not always hold. For example, consider the sentence from above *"My friend, a talented musician, is coming over tonight."* The information the friend is a talented musician can be a very crucial bit of information depending on the context (e.g., when organizing a party and to provide some form of entertainment). While the simplified sentence *"My friend is coming over tonight."* is perfectly correct, the friend can now be any friend with no indication that he or she can help with the entertainment.

---

## Summary

Dependency parsing is a natural language processing technique that analyzes the grammatical structure of sentences. It identifies the relationships between words, represented as a directed graph called a dependency tree. Each word is considered a node, and the relationships between words are depicted as directed edges. Dependency parsing helps understand the syntactic dependencies, such as subject-verb and verb-object relationships, providing insights into the sentence's structure and meaning. It is used in various NLP tasks, such as information extraction, question answering, and machine translation.

By utilizing dependency parsing, we can extract meaningful insights from text and represent them in a structured format. This allows for efficient information retrieval and analysis. Dependency parsing helps uncover the relationships between words, empowering us to navigate through the syntactic structure of sentences and identify dependencies accurately. It serves as a foundation for building intelligent systems that can understand and process human language, contributing to tasks such as semantic search, sentiment analysis, and text summarization.